# OpenMotion AI - Backend com LTX-Video (Muito mais rápido)
Versão otimizada para 720p + motion control forte usando LTX-Video + IC-LoRA Pose

In [ ]:
!nvidia-smi

import os
import shutil
os.chdir('/content')
if os.path.exists('ltx_backend'):
    shutil.rmtree('ltx_backend')

!git clone https://github.com/Lightricks/LTX-Video.git ltx_backend
os.chdir('/content/ltx_backend')

# Instalações otimizadas para T4
!pip install -U diffusers transformers accelerate xformers torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install mediapipe opencv-python pillow fastapi uvicorn python-multipart pyngrok

import torch
torch.backends.cudnn.benchmark = True
print("✅ Ambiente pronto")

### 2. Criar os arquivos do backend

In [ ]:
%%writefile /content/ltx_backend/backend/main.py
import os
import torch
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
import uuid
import shutil
from pathlib import Path
from pipeline_ltx import LTXPosePipeline  # arquivo que vamos criar abaixo
import asyncio

app = FastAPI(title="OpenMotion AI - LTX Backend")

# Carrega o pipeline UMA ÚNICA VEZ (muito importante para velocidade)
print("Carregando LTX-Video + Pose IC-LoRA (pode demorar 40-70s)...")
global_pipeline = LTXPosePipeline()
print("✅ Pipeline carregado")

@app.post("/generate")
async def generate(
    character_image: UploadFile = File(...),
    reference_video: UploadFile = File(...),
    prompt: str = Form("a person dancing, high quality, detailed"),
    negative_prompt: str = Form("blurry, low quality, deformed")
):
    task_id = str(uuid.uuid4())[:8]
    temp_dir = Path(f"/content/temp_{task_id}")
    temp_dir.mkdir(exist_ok=True)

    try:
        # Salva arquivos
        char_path = temp_dir / "character.png"
        vid_path = temp_dir / "reference.mp4"

        with open(char_path, "wb") as f:
            f.write(await character_image.read())
        with open(vid_path, "wb") as f:
            f.write(await reference_video.read())

        # Gera o vídeo (mesma lógica do seu frontend)
        output_path = await global_pipeline.generate(
            character_path=str(char_path),
            reference_video=str(vid_path),
            prompt=prompt,
            negative_prompt=negative_prompt,
            output_dir=str(temp_dir)
        )

        return FileResponse(output_path, media_type="video/mp4", filename=f"dance_{task_id}.mp4")

    finally:
        # Limpeza
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
%%writefile /content/ltx_backend/backend/pipeline_ltx.py
import torch
from diffusers import LTXPipeline  # ou LTXConditionPipeline conforme versão
from PIL import Image
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
import subprocess

class LTXPosePipeline:
    def __init__(self):
        self.device = "cuda"
        # Carrega modelo base LTX + IC-LoRA Pose Control
        self.pipe = LTXPipeline.from_pretrained(
            "Lightricks/LTX-Video",  # ou LTX-2 se preferir versão mais nova
            torch_dtype=torch.bfloat16,
            variant="fp16"
        ).to(self.device)

        # Carrega LoRA de Pose (IC-LoRA)
        self.pipe.load_lora_weights(
            "Lightricks/LTX-Video-ICLoRA-pose-13b-0.9.7",
            weight_name="ltxv-097-ic-lora-pose-control-diffusers.safetensors"
        )
        self.pipe.fuse_lora()

        self.pipe.enable_xformers_memory_efficient_attention()
        self.pipe.enable_vae_slicing()

        # MediaPipe para extrair pose
        self.mp_pose = mp.solutions.pose
        self.pose_detector = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,   # 1 = mais rápido
            enable_segmentation=False
        )

    async def generate(self, character_path, reference_video, prompt, negative_prompt, output_dir):
        # 1. Extrai frames e poses do vídeo de referência
        cap = cv2.VideoCapture(reference_video)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = []
        pose_images = []

        while len(frames) < 60:  # limite ~2-4 segundos por chunk (ajuste se quiser mais)
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))

            # Extrai pose
            results = self.pose_detector.process(rgb)
            if results.pose_landmarks:
                # Aqui você pode desenhar o esqueleto ou usar como condição extra
                pass
            pose_images.append(rgb)  # simplificado - LTX usa o vídeo como referência + LoRA

        cap.release()

        character_img = Image.open(character_path).convert("RGB")

        # Geração com LTX + Pose Control
        output = self.pipe(
            image=character_img,                    # personagem novo
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=20,                 # rápido
            guidance_scale=6.0,
            height=720,
            width=1280,
            num_frames=len(frames),
            fps=24,                                 # nativo 24-30 FPS
            generator=torch.Generator(device="cuda").manual_seed(42),
        )

        video_path = Path(output_dir) / "output.mp4"
        # Exporta vídeo (use o export_to_video do diffusers ou ffmpeg)
        output.frames[0].save(str(video_path), fps=24)

        return str(video_path)